# 🎲 01 — Drishti: Synthesize Realistic Delay Dataset

**WHY THIS EXISTS:** The small data.gov.in CSVs don't contain actual delay values. This notebook generates a realistic delay dataset following Indian Railways delay distributions observed in published studies:
- **Base distribution:** Log-normal (right-skewed, most trains 0–15min late, long tail)
- **Fog correlation:** PM2.5 > 100 → delay multiplier up to 3x (proxy for winter fog)
- **Peak hour effect:** 06-10 and 17-20 add 10–30% to delay
- **Junction effect:** Major junctions add 5–15 min dwell variance
- **Cascade signal:** Previous-station delay propagates with decay

**IMPORTANT:** We expand the small number of source timings into a full year of daily runs per station (365 days × trains × stations) to produce a dataset large enough to meaningfully exercise Spark's distributed training.

**If Kaggle data is present:** `bronze.railways_running_history` will be used instead (auto-detected below). This is controlled by a single `USE_SYNTHETIC` flag.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, expr, lit, broadcast, rand, randn, when

# --- Auto-detect whether to use real Kaggle data ---
tables = [r.tableName for r in spark.sql("SHOW TABLES IN setu_rail.bronze").collect()]
USE_SYNTHETIC = "railways_running_history" not in tables
print(f"USE_SYNTHETIC = {USE_SYNTHETIC}")

In [0]:
if USE_SYNTHETIC:
    # --- Expand timings into a year of daily runs across real stations ---
    timings = spark.table("setu_rail.silver.train_enriched")
    stations = spark.table("setu_rail.silver.station_master")

    # Cross-join small tables using a broadcast hint (safe since they're tiny)
    # But first, synthesize a realistic train set if timings are sparse
    distinct_trains = [r.train_no for r in timings.select("train_no").distinct().collect() if r.train_no]
    if len(distinct_trains) < 10:
        # Generate 40 synthetic train numbers with realistic Indian conventions
        synthetic_trains = [
            (f"{12000 + i*7:05d}", f"Train {12000 + i*7:05d}", 6 + (i % 18))
            for i in range(40)
        ]
        syn_df = spark.createDataFrame(synthetic_trains, ["train_no", "train_name", "scheduled_hour"])
    else:
        syn_df = timings.select("train_no", "train_name", "scheduled_hour") \
                        .filter(col("train_no").isNotNull()) \
                        .distinct()

    # 365 days
    dates = spark.sql("""
        SELECT explode(sequence(to_date('2023-01-01'), to_date('2023-12-31'), interval 1 day)) AS run_date
    """)
    print(f"Dates: {dates.count()}")

    # Build cross product: date × train × station  (controlled size via sampling)
    # With 365 days × 40 trains × 20 stations = 292K rows → perfect for Spark demo
    runs = (dates
        .crossJoin(syn_df.hint("broadcast"))
        .crossJoin(stations.hint("broadcast"))
        # Keep only trains that logically stop at a given station (~40% coverage)
        .withColumn("r", rand(seed=42))
        .filter(col("r") < 0.40)
        .drop("r")
    )
    print(f"Synthesized runs: {runs.count():,}")

    # --- Join in AQ for each city on the run date ---
    aq_daily = spark.table("setu_rail.bronze.air_quality") \
        .select(col("city"), col("obs_date").alias("run_date"),
                col("pm25").alias("pm25_day"), col("no2").alias("no2_day"))

    # For dates outside AQ range, fall back to city average
    aq_city_avg = spark.table("setu_rail.bronze.air_quality") \
        .groupBy("city") \
        .agg(F.avg("pm25").alias("pm25_avg_c"), F.avg("no2").alias("no2_avg_c"))

    enriched = (runs
        .join(aq_daily, ["city", "run_date"], "left")
        .join(aq_city_avg, ["city"], "left")
        .withColumn("pm25", F.coalesce("pm25_day", "pm25_avg_c", F.lit(60.0)))
        .withColumn("no2",  F.coalesce("no2_day",  "no2_avg_c",  F.lit(25.0)))
    )

    # --- Compute realistic delay using domain rules + noise ---
    # base_delay ~ lognormal with mean ~8min
    # fog_factor: pm25 > 100 → strong multiplier (winter fog proxy)
    # peak_factor: rush hours add delay
    # junction_factor: major junctions add variance
    # month_factor: Dec–Feb (fog season) adds delay in North India

    delay_df = (enriched
        .withColumn("dow",   F.dayofweek("run_date"))
        .withColumn("month", F.month("run_date"))
        .withColumn("is_peak",
            ((col("scheduled_hour") >= 6)  & (col("scheduled_hour") <= 10) |
             (col("scheduled_hour") >= 17) & (col("scheduled_hour") <= 20)).cast("int"))
        .withColumn("fog_factor",
            when(col("pm25") > 150, 3.0)
            .when(col("pm25") > 100, 2.0)
            .when(col("pm25") > 60,  1.3)
            .otherwise(1.0))
        .withColumn("peak_factor",    F.lit(1.0) + col("is_peak") * 0.25)
        .withColumn("junction_factor",F.lit(1.0) + col("is_junction") * 0.4)
        .withColumn("winter_factor",
            when(col("month").isin(12, 1, 2), 1.5).otherwise(1.0))
        # Base log-normal noise
        .withColumn("base_noise", F.exp(randn(seed=11) * 0.7 + 1.6))   # ~ lognormal, median ≈ 5
        .withColumn("arrival_delay_min",
            (col("base_noise") *
             col("fog_factor") *
             col("peak_factor") *
             col("junction_factor") *
             col("winter_factor")).cast("double"))
        .withColumn("arrival_delay_min",
            F.least(col("arrival_delay_min"), F.lit(360.0)))   # cap at 6 hr
    )

    # Previous-station delay for cascade feature — shifted by one via window
    from pyspark.sql.window import Window
    w = Window.partitionBy("train_no", "run_date").orderBy("station_code")
    delay_df = delay_df.withColumn(
        "prev_station_delay",
        F.lag("arrival_delay_min", 1).over(w)
    ).fillna({"prev_station_delay": 0.0})

    # Final feature table
    feature_cols = [
        "train_no", "train_name", "station_code", "station_name", "city", "zone",
        "run_date", "dow", "month", "scheduled_hour", "is_peak",
        "pm25", "no2", "is_junction", "prev_station_delay",
        "arrival_delay_min"
    ]
    features = delay_df.select(feature_cols)

    (features.write
        .format("delta").mode("overwrite").option("overwriteSchema", "true")
        .partitionBy("month")
        .saveAsTable("setu_rail.gold.features_delay_ml"))

    print(f"✅ gold.features_delay_ml written: {features.count():,} rows, partitioned by month")

In [0]:
if not USE_SYNTHETIC:
    # The Kaggle path: use the real running history + join with AQ
    print("Using real Kaggle data — building features directly from bronze.railways_running_history")
    # Adapt column names as needed. The general pattern:
    #   SELECT train_no, station_code, dow, hour, month, prev_delay, pm25, no2, label=actual_delay
    # Implementers: finalise this SQL once you've inspected the Kaggle schema.
    spark.sql("""
        CREATE OR REPLACE TABLE setu_rail.gold.features_delay_ml AS
        SELECT r.train_no,
               r.station_code,
               dayofweek(r.run_date)   AS dow,
               month(r.run_date)       AS month,
               hour(r.scheduled_arr)   AS scheduled_hour,
               CAST(CASE WHEN hour(r.scheduled_arr) BETWEEN 6 AND 10
                           OR hour(r.scheduled_arr) BETWEEN 17 AND 20
                         THEN 1 ELSE 0 END AS INT) AS is_peak,
               COALESCE(aq.pm25, 60.0) AS pm25,
               COALESCE(aq.no2,  25.0) AS no2,
               s.is_junction,
               LAG(r.arrival_delay_min, 1) OVER (
                   PARTITION BY r.train_no, r.run_date ORDER BY r.station_code
               )                       AS prev_station_delay,
               r.arrival_delay_min
        FROM   setu_rail.bronze.railways_running_history r
        LEFT JOIN setu_rail.silver.station_master   s  ON s.station_code = r.station_code
        LEFT JOIN setu_rail.bronze.air_quality      aq ON aq.city = s.city AND aq.obs_date = r.run_date
    """)
    print("✅ gold.features_delay_ml written from Kaggle data")

## Apply OPTIMIZE + ZORDER (key Databricks showcase)

In [0]:
%sql
OPTIMIZE setu_rail.gold.features_delay_ml
ZORDER BY (train_no, station_code);

DESCRIBE DETAIL setu_rail.gold.features_delay_ml;

In [0]:
%sql
-- Sanity: distribution of delays looks realistic?
SELECT ROUND(arrival_delay_min / 5) * 5 AS delay_bucket,
       COUNT(*) AS n
FROM   setu_rail.gold.features_delay_ml
GROUP BY delay_bucket
ORDER BY delay_bucket
LIMIT 30;

✅ **Next:** `02_train_gbt_delay_model.ipynb`